# Case set up
### Import libraries

In [1]:
import os
import sys
import copy
import time
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib as jl
threads = 4

from tqdm import tqdm
from sklearn.model_selection import train_test_split

# tf.config.threading.set_inter_op_parallelism_threads(1)
# tf.config.threading.set_intra_op_parallelism_threads(1)

Visual set up

In [2]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
%matplotlib qt5

# Use custom style
style_name = 'paper'
mpl.style.use(
    os.environ['WORKSPACE_PATH'] + '/RESEARCH/styles/matplotlib/' + style_name + '.mplstyle'
) # See: https://matplotlib.org/1.5.3/users/style_sheets.html

In [3]:
# Get colors list
colors = mpl.rcParams['axes.prop_cycle'].by_key()['color']*10

# Figure/animation saving parameters
# mpl.rcParams['savefig.dpi'] = 100

# Font parameters
mpl.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# mpl.rcParams['text.latex.preamble'] = join([
#     r'\usepackage{amsmath}',
#     r'\boldmath',
#     r'\usepackage{pifont}',
#     r'\usepackage{tikz}',
#     r'\usepackage[utf8]{inputenc}'
# ])
# mpl.rcParams['font.family'] = 'lmodern'

# # Tick parameters
# mpl.rcParams['xtick.labelsize'] = 40
# mpl.rcParams['ytick.labelsize'] = 40
# mpl.rcParams['xtick.major.size'] = 20
# mpl.rcParams['xtick.major.width'] = 2
# mpl.rcParams['xtick.minor.size'] = 10
# mpl.rcParams['xtick.minor.width'] = 2

# # Tick parameters
# mpl.rcParams['axes.labelsize'] = 40
# mpl.rcParams['axes.titlesize'] = 40
# mpl.rcParams['legend.fontsize'] = 35


### Define PrODE and case (model) directories paths

In [4]:
prode_dir = os.environ['WORKSPACE_PATH'] + '/ML/PrODE/prode/'
case_dir = "./"

In [5]:
if importlib.util.find_spec('prode') is None:
    sys.path.append(os.path.normpath(prode_dir))
sys.path.append(os.path.normpath(case_dir))
cwd = os.getcwd()

### Import additional libraries

In [6]:
# PrODE -----------------------------------------------------------------------
from prode.utilities import utils
from prode.apps import functionObjects as fun
from prode.postprocessing import testing, training

# Extra modules ---------------------------------------------------------------
# Inputs
import inputs
# Physics-related
from physics.data_generator import DataGenerator
from prode.physics.general import const
from prode.physics.thermochemistry import specie as specieMdl
from prode.physics.thermochemistry.reactionThermo import mixture as mixMdl

Change directory

In [7]:
os.chdir(os.path.normpath(case_dir))

### Initialize network

In [8]:
net = fun._net(
    input_module=inputs
)


[PrODE]: Initializing 'NoneqControllerNN' network ...


### Initialize PrODE model

In [9]:
from inputs.general import flags

flags['load'] = 1

for k in ('train', 'save'):
    flags[k] = 0
if flags['load'] not in (1, 2):
    utils.raise_value_err(
        "The network has to be restored from a saved or a checkpointed "
        "one. Modified the 'load' entry in the `flags` input variable."
    )
model = fun._model(
    flags=flags,
    input_module=inputs,
    net=net
)


[PrODE]: Building 'NoneqControllerNN' network ...
Model: "NonEqLevel_MMOpNN"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 29)]         0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 1)]          0                                            
__________________________________________________________________________________________________
branch_fn (Lambda)              (None, 29)           0           input_1[0][0]                    
__________________________________________________________________________________________________
trunk_fn (Scaling)              (None, 1)            2           input_2[0][0]                    
_______________________________

### Generate testing data
Choose specific test scenarios

In [10]:
indices = np.array([1]).T

In [11]:
# Generate Testing Data ---------------------------------------------------
dataGen_args = utils.get_args_values(
    args=utils.get_args(DataGenerator.__init__),
    modules=[inputs.data, inputs.general]
)
dataGen = DataGenerator(
    **dataGen_args
)
dataGen.data_dir = case_dir + "/data/"
if not os.path.exists(dataGen.data_dir):
    os.makedirs(dataGen.data_dir)

utils.print_main("Generating testing pts ...")
test_data = dataGen.testing_data(
    inputs.data.file['test'],
    header=inputs.data.header,
    indices=indices-1
)

100%|██████████| 1/1 [00:00<00:00, 37.62it/s]


[PrODE]: Generating testing pts ...


Change directory

In [12]:
os.chdir(os.path.normpath(cwd))

# Postprocessing - Model analysis

## Set up analysis

In [13]:
path = cwd + '/outputs/'
if not os.path.exists(path):
    os.makedirs(path)
    
FLAGS = {
    'pred': 1,
    'input_space': 0,
    'one_pass': 0,
    'accuracy': 0,
    'comp_time': 1,
    'speedup': 0,
    'recursive': 0,
    'noneq_dist': 0,
    'levels': 0,
    'animations': 0,
    'adaptive': 0,
    'park_ts1': 0
}

#### Get predictions

In [14]:
if FLAGS['pred']:
    utils.print_main("Postprocessing physical quantities ...")
    x, y, y_pred = [], [], []
    for scenario in tqdm(test_data):
        x_i, y_i = scenario
        x.append(x_i)
        y.append(y_i)
        y_pred.append(model.net.call(x_i))
    N = len(test_data)

  0%|          | 0/1 [00:00<?, ?it/s]


[PrODE]: Postprocessing physical quantities ...


100%|██████████| 1/1 [00:00<00:00,  3.96it/s]


In [15]:
tol = 0.5
tf.nest.map_structure(lambda x: x>tol, y_pred[0])

[<tf.Tensor: shape=(1418, 1), dtype=bool, numpy=
 array([[ True],
        [ True],
        [ True],
        ...,
        [False],
        [False],
        [False]])>,
 <tf.Tensor: shape=(1418, 3), dtype=bool, numpy=
 array([[ True,  True,  True],
        [ True,  True,  True],
        [ True,  True,  True],
        ...,
        [ True,  True,  True],
        [ True,  True,  True],
        [ True,  True,  True]])>,
 <tf.Tensor: shape=(1418, 9), dtype=bool, numpy=
 array([[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        ...,
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]])>]

In [20]:
model.net.call(x[0], training=False, tol=0.5)

[<tf.Tensor: shape=(1418, 1), dtype=float64, numpy=
 array([[0.64008968],
        [0.64008968],
        [0.64008968],
        ...,
        [0.43611363],
        [0.43611355],
        [0.43611348]])>,
 <tf.Tensor: shape=(1418, 3), dtype=float64, numpy=
 array([[0.76703679, 0.62928599, 0.95750985],
        [0.76703679, 0.62928599, 0.95750984],
        [0.76703679, 0.62928599, 0.95750984],
        ...,
        [0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ]])>,
 <tf.Tensor: shape=(1418, 9), dtype=float64, numpy=
 array([[4.45733472, 1.21839947, 0.83974756, ..., 0.73580109, 0.95380947,
         1.15423875],
        [4.45733471, 1.21839947, 0.83974756, ..., 0.73580109, 0.95380947,
         1.15423875],
        [4.4573347 , 1.21839947, 0.83974756, ..., 0.73580109, 0.95380947,
         1.15423875],
        ...,
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
       